## Labelled-data curve

How much of the fine-tuning gain survives on less labelled data. RoBERTa-large is
refit on stratified subsets of each training partition and scored on the same
untouched test split. The 1,984 point is the existing `roberta-large` row.


### Colab Setup

In [ ]:
import os
import subprocess
import sys

# local runs: the repo root is one level up. Colab chdirs there below.
sys.path.insert(0, "..")

# On Colab: clone the repo, install deps, mount Drive for results.csv. The repo is
# public, so no token. Python caches imports -- restart the runtime after any code
# change, or the clone refreshes and the old module stays loaded.
REPO = "https://github.com/IronQuant/mlds_codebase.git"
ROOT = "/content/mlds_codebase"

if "google.colab" in sys.modules:
    if os.path.isdir(ROOT):
        subprocess.run(["git", "-C", ROOT, "fetch", "-q", "origin"], check=True)
        subprocess.run(
            ["git", "-C", ROOT, "reset", "--hard", "-q", "origin/main"], check=True
        )
    else:
        subprocess.run(["git", "clone", "-q", REPO, ROOT], check=True)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "transformers>=4.48",
            "ftfy",
            "nltk",
            "polars",
            "fastexcel",
            "sentencepiece",
            "protobuf",
        ],
        check=True,
    )
    os.chdir(ROOT)
    sys.path.insert(0, ROOT)

    from google.colab import drive

    drive.mount("/content/drive")

### Key Imports

In [ ]:
import polars as pl
import torch

from config import RESULTS_DIR, SHAH_PLM, SHAH_SEEDS
from data.loader_twd_labelled import load_splits
from models.plm_finetune import finetune

from utils.results import already_done, save_result

OUT = RESULTS_DIR / "results.csv"
ENC = "roberta-large"
SEEDS = SHAH_SEEDS
SIZES = (125, 250, 500, 1000)
FORCE = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("results ->", OUT, "| device:", DEVICE)
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))


### Subsample

Stratified by label, so no class drops out at the small end. The test split is
never touched, only the training half shrinks.


In [ ]:
import polars as pl; print(pl.__version__)
def subsample(train, n, seed):
    """
    Subsample the training data to have approximately n examples,
    Maintaining class balance.
    Args:
        train (pl.DataFrame): The training data.
        n (int): The desired number of examples.
        seed (int): Random seed for reproducibility.

    Returns:
        pl.DataFrame: The subsampled training data.
    """
    parts = []
    # narrow first: older polars aggregates every column here and trips
    # on the loader's index columns
    train = train.select("sentence", "label")
    for g in train.partition_by("label"):
        k = max(1, round(n * len(g) / len(train)))
        parts.append(g.sample(n=min(k, len(g)), shuffle=True, seed=seed))
    return pl.concat(parts)


# class balance at the small end
train, _ = load_splits("benchmark", seed=SEEDS[0])
for n in SIZES:
    s = subsample(train, n, SEEDS[0])
    print(n, "->", len(s), "|", s["label"].value_counts().sort("label").to_dicts())


### Fine-tune

`finetune()` carves 20% of what it is handed for validation, so at n=125 early
stopping runs off 25 examples. We accept that noise rather than hold the
validation set fixed, since instability at small labelled counts is part of what
this curve measures.


In [ ]:
cfg = SHAH_PLM[ENC]

for n in SIZES:
    for seed in SEEDS:
        model_key = f"subset-{n}:{ENC}"
        if already_done(OUT, force=FORCE, model=model_key, corpus="twd", seed=seed):
            print(f"{model_key} seed {seed}: already done, skipping")
            continue
        train, test = load_splits("benchmark", seed=seed)
        small = subsample(train, n, seed)
        print(f"{model_key} seed {seed}: {len(small)} train rows", flush=True)
        model, tok_, metrics = finetune(
            small,
            model_name=cfg["model_name"],
            lr=cfg["lr"],
            batch_size=cfg["batch_size"],
            seed=seed,
            test_df=test.select("sentence", "label"),
            device=DEVICE,
            verbose=True,
        )
        save_result(
            OUT,
            model=model_key,
            corpus="twd",
            seed=seed,
            epochs=metrics["epochs"],
            weighted_f1=round(metrics["test_f1"], 4),
            macro_f1=round(metrics["test_macro_f1"], 4),
        )
        print(f"{model_key} seed {seed}: macro={metrics['test_macro_f1']:.4f}")
        del model, tok_
        torch.cuda.empty_cache()


### Curve

The 1,984 endpoint comes from the existing `roberta-large` rows, so it is the
same run reported in the approach comparison.


In [ ]:
d = (
    pl.read_csv(OUT)
    .filter(pl.col("corpus") == "twd")
    .filter(pl.col("model").str.starts_with("subset-") | (pl.col("model") == ENC))
)
for m in sorted(d["model"].unique()):
    v = d.filter(pl.col("model") == m)["macro_f1"]
    print(f"{m:26s} mean {v.mean():.4f}  sd {v.std(ddof=0):.4f}  n {len(v)}")
